In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [2]:
GEN_1_RANDOM = CF_OUTPUTS / "gen1_models_explainers_constraints" / "random_search_exp"
GEN_1_RANDOM.is_dir()

True

In [3]:
batch_data = GEN_1_RANDOM / "RF_prange_highthres_2026-04-29" / "annotated.csv"
batch_df = pd.read_csv(batch_data)

### set PD options

In [4]:
pd.set_option("display.max_rows", None)

In [5]:
# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [6]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

In [7]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,99.45,,,,0.0633,
1,0,cf_1,,,,,,,16.9409,3,,0.1306,2,False,0.0633,0.0967
2,0,cf_2,,,,,,,15.2291,6,,0.25,2,False,0.0633,0.17
3,0,cf_3,2,2,,,,,,,,0.25,2,False,0.0633,0.1267
4,0,cf_4,,,,,,,17.7551,,,0.041,1,False,0.0633,0.0467
5,0,cf_5,,,,,,,16.5046,,,0.0826,1,False,0.0633,0.0933
6,0,cf_6,,,5,,,,16.1725,,,0.2186,2,False,0.0633,0.0767
7,0,cf_7,,2,5,,,,,,,0.25,2,False,0.0633,0.0533
8,0,cf_8,,,,,,,18.2293,,,0.0252,1,False,0.0633,0.0433
9,0,cf_9,,2,,,,,17.0718,,,0.1887,2,False,0.0633,0.16


In [8]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation
# prefers valid CFs without violations, and lowest Gower
single_cf_df = select_one_cf_per_query(batch_df)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,99.45,,,,0.0633,
9,0,cf_1,,,,,,,16.9409,3,,0.1306,2,False,0.0633,0.0967
1,1,xin,3,4,1,2,3,0,22.3757,0,98.56,,,,0.1467,
10,1,cf_2,,,,4,,,19.2603,,,0.1298,2,True,0.1467,0.0267
2,2,xin,5,3,1,1,4,0,22.694,7,133.88,,,,0.1567,
11,2,cf_5,2,,,,,,,,,0.125,1,True,0.1567,0.0733
3,3,xin,3,3,6,6,2,0,24.3418,1,97.63,,,,0.02,
12,3,cf_1,,,,,,,20.2869,2,,0.1886,2,False,0.02,0.08
4,4,xin,5,4,2,7,1,0,21.2585,3,99.39,,,,0.0267,
13,4,cf_1,,,,,,,21.0769,,,0.0037,1,False,0.0267,0.0167
